# Baseline Comparison

Three baselines compared against the full Two-Stage Hybrid Ensemble.

| Baseline | Description |
|---|---|
| Naive Mean | Predict training-set mean for every test observation |
| Probit (Yield Curve) | Linear regression on 10yr − 3mo spread only |
| Single-Stage XGBoost | Plain XGBoost on all features, no chain, no ensemble |
| **Ours** | Full Chain Stacking (CatBoost + LightGBM + RF + ElasticNet) |

**Train/test split:** pre-2020-01-01 (635 obs) / post-2020-01-01 (65 obs)  
**Data:** `data/fix/feature_selected_reg_full.csv`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import DMatrix, train as xgb_train
import warnings
warnings.filterwarnings('ignore')

TARGETS = [
    'recession_probability',
    '1_month_recession_probability',
    '3_month_recession_probability',
    '6_month_recession_probability',
]
LABELS = ['Current', '1-Month', '3-Month', '6-Month']

OURS = {
    'Current': {'MAE': 6.8292, 'RMSE': 22.5250},
    '1-Month': {'MAE': 5.6336, 'RMSE': 16.0698},
    '3-Month': {'MAE': 7.7285, 'RMSE': 20.3876},
    '6-Month': {'MAE': 10.1696, 'RMSE': 21.2089},
}

df = pd.read_csv('../data/fix/feature_selected_reg_full.csv')
df['date'] = pd.to_datetime(df['date'])

train_df = df[df['date'] < '2020-01-01'].copy()
test_df  = df[df['date'] >= '2020-01-01'].copy()

def clean(d):
    d = d.replace([np.inf, -np.inf], np.nan)
    return d.ffill().bfill().fillna(0)

X_train = clean(train_df.drop(columns=TARGETS + ['date']))
X_test  = clean(test_df.drop(columns=TARGETS + ['date']))
y_train = clean(train_df[TARGETS])
y_test  = clean(test_df[TARGETS])

EPS = 1e-6
def logit(y):
    y_s = np.clip(np.array(y, dtype=float) / 100.0, EPS, 1 - EPS)
    return np.log(y_s / (1 - y_s))

def inv_logit(z):
    return 1.0 / (1.0 + np.exp(-np.array(z, dtype=float))) * 100.0

def metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse

print(f'Train: {len(X_train)} | Test: {len(X_test)} | Features: {X_train.shape[1]}')

In [ ]:
# ── Baseline 2: Naive Mean ──────────────────────────────────
print('=== Baseline 2: Naive Mean Predictor ===')
naive = {}
for t, lbl in zip(TARGETS, LABELS):
    train_mean = y_train[t].mean()
    preds = np.full(len(y_test), train_mean)
    mae, rmse = metrics(y_test[t].values, preds)
    naive[lbl] = {'MAE': mae, 'RMSE': rmse, 'pred': preds}
    print(f'  {lbl:10s}  train_mean={train_mean:.3f}  MAE={mae:.4f}  RMSE={rmse:.4f}')

In [ ]:
# ── Baseline 1: Probit / Yield Curve Spread ──────────────────
print('=== Baseline 1: Linear Regression — Yield Curve Spread ===')
spread_train = (X_train['10_year_rate'] - X_train['3_months_rate']).values.reshape(-1, 1)
spread_test  = (X_test['10_year_rate']  - X_test['3_months_rate']).values.reshape(-1, 1)

probit = {}
for t, lbl in zip(TARGETS, LABELS):
    reg = LinearRegression()
    reg.fit(spread_train, logit(y_train[t].values))
    preds = np.clip(inv_logit(reg.predict(spread_test)), 0, 100)
    mae, rmse = metrics(y_test[t].values, preds)
    probit[lbl] = {'MAE': mae, 'RMSE': rmse, 'pred': preds}
    print(f'  {lbl:10s}  coef={reg.coef_[0]:+.4f}  MAE={mae:.4f}  RMSE={rmse:.4f}')

In [ ]:
# ── Baseline 3: Single-Stage XGBoost ──────────────────────────
print('=== Baseline 3: Single-Stage XGBoost (no chain, no ensemble) ===')
XGB_PARAMS = {
    'objective': 'reg:squarederror', 'max_depth': 5, 'eta': 0.05,
    'subsample': 0.9, 'colsample_bytree': 0.9, 'seed': 42, 'verbosity': 0,
}

singlexgb = {}
for t, lbl in zip(TARGETS, LABELS):
    dtrain = DMatrix(X_train.values, label=logit(y_train[t].values))
    dtest  = DMatrix(X_test.values)
    model  = xgb_train(XGB_PARAMS, dtrain, num_boost_round=500)
    preds  = np.clip(inv_logit(model.predict(dtest)), 0, 100)
    mae, rmse = metrics(y_test[t].values, preds)
    singlexgb[lbl] = {'MAE': mae, 'RMSE': rmse, 'pred': preds}
    print(f'  {lbl:10s}  MAE={mae:.4f}  RMSE={rmse:.4f}')

In [ ]:
# ── Final comparison table ──────────────────────────────────
print('=' * 76)
print(f'{"Model":<30} {"MAE Curr":>10} {"MAE 1M":>10} {"MAE 3M":>10} {"MAE 6M":>10}')
print('-' * 76)
for name, res in [
    ('Naive Mean',           naive),
    ('Probit (Yield Curve)', probit),
    ('Single-Stage XGBoost', singlexgb),
]:
    print(f'{name:<30} {res["Current"]["MAE"]:>10.4f} {res["1-Month"]["MAE"]:>10.4f} {res["3-Month"]["MAE"]:>10.4f} {res["6-Month"]["MAE"]:>10.4f}')
print(f'{"Ours (Two-Stage Ensemble)":<30} {OURS["Current"]["MAE"]:>10.4f} {OURS["1-Month"]["MAE"]:>10.4f} {OURS["3-Month"]["MAE"]:>10.4f} {OURS["6-Month"]["MAE"]:>10.4f}')
print('=' * 76)